# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
from datetime import datetime as dt

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

from matplotlib.figure import Figure

import panel as pn

from enderleaf.const import (
    ImageMergeMode,
    ImageMergeMethod,
    TIME_FORMAT,
    PRECISE_TIME_FORMAT,
    DEFAULT_DATETIME_FORMAT,
    COLOR_SPACES,
)
from enderleaf.draw import image_grid, concat_tile_resize, plot_images_with_histograms
from enderleaf.tools import read_dataframe, write_dataframe, format_datetime, ensure_folder
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    merge_images_channels,
    match_previous_rotation,
    get_circles,
    get_channels,
    get_channel,
    equalize_hist,
)
from enderleaf.draw import draw_circles

In [ ]:
pn.extension("ipywidgets")

## Constants

In [ ]:
EXP = "Exp26DM02"
INOC = "I1"
# PLATE = 0
MONTH = 5
DAY = 26

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
PATH_TO_PPIMAGES = Path(".").joinpath("output", "pre_processed", EXP, INOC)
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
).dropna(subset="north")
# df = df[df.plate == PLATE]
df = df[df.month == MONTH]
df = df[df.day == DAY]
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df["file_name"].apply(lambda x:PATH_TO_IMAGES.joinpath(x).is_file())
df = df[df.file_ok == True]
df = df[df.job_ts != 20260515164717]
df["leaf_id"] = df.plate.astype(str)+df.row.astype(str)+df.col.astype(str)
df

In [ ]:
pd.DataFrame(df.groupby(["job_ts", "plate", "light_cycle", "card_count"]).height.mean()).reset_index().sort_values("plate")

## Select Cycle ID

In [ ]:
sel_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

sel_color_space = pn.widgets.Select(
    name="Color space",
    options=["rgb", "hsv", "lab", "yuv", "ycrcb"],
    sizing_mode="scale_width",
    value="rgb",
)
sel_channel_1 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][0],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_2 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][1],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_channel_3 = pn.widgets.Select(
    name=COLOR_SPACES[sel_color_space.value][2],
    options={
        k.name: k
        for k in [
            ImageMergeMode.MIN,
            ImageMergeMode.MAX,
            ImageMergeMode.MEDIAN,
            ImageMergeMode.AVG,
        ]
    },
    sizing_mode="scale_width",
)
sel_merge_method = pn.widgets.Select(
    name="Merge method",
    options={
        k.name: k
        for k in [
            ImageMergeMethod.RGB,
            ImageMergeMethod.HSV,
            ImageMergeMethod.LAB,
            ImageMergeMethod.YUV,
            ImageMergeMethod.YCrCb,
        ]
    },
    sizing_mode="scale_width",
)
bt_random = pn.widgets.Button(name="Random Disc")

img_out = pn.pane.Matplotlib(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df[
        (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]

        sel_plate.value = row.plate
        sel_row.value = row.row
    finally:
        updating = False
    sel_col.value = row.col


bt_random.on_click(on_random)

@pn.depends(sel_merge_method.param.value, watch=True)
def on_merge_method_changed(mm):
    global updating
    updating = True
    try:
        sel_color_space.value = mm.value[0]
        sel_channel_1.value = mm.value[1][0]
        sel_channel_2.value = mm.value[1][1]
    finally:
        updating = False
    sel_channel_3.value = mm.value[1][2]


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel_1.name = COLOR_SPACES[cs][0]
    sel_channel_2.name = COLOR_SPACES[cs][1]
    sel_channel_3.name = COLOR_SPACES[cs][2]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_plate,
            sel_row,
            sel_col,
            sel_color_space,
            sel_channel_1,
            sel_channel_2,
            sel_channel_3,
        ]
    ],
    watch=True,
)
def on_ld_changed(plate, row, col, cs, cn1, cn2, cn3):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    merged_images = []
    card_counts = []
    for card_count in df_ld.card_count.unique():
        image_list = [
            crop_image(load(row[1]), crop_data)
            for row in df_ld[df_ld.card_count == card_count].iterrows()
        ]
        merged_images.append(
            merge_images_channels(
                image_list=image_list, color_space=cs, merge_modes=(cn1, cn2, cn3)
            )
        )
        card_counts.append(f"{card_count} {len(image_list)}")
    img_out.object = plot_images_with_histograms(
        images=merged_images, color_spaces=[cs, "rgb"], titles=card_counts
    )


on_ld_changed(
    sel_plate.value,
    sel_row.value,
    sel_col.value,
    sel_color_space.value,
    sel_channel_1.value,
    sel_channel_2.value,
    sel_channel_3.value,
)

pn.Column(
    pn.Row(
        sel_plate,
        sel_row,
        sel_col,
        bt_random,
        sel_merge_method,
        sel_color_space,
        sel_channel_1,
        sel_channel_2,
        sel_channel_3,
    ),
    pn.Row(img_out),
)

In [ ]:
df_cc_1 = (
    pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")])
    .sort_values(["plate", "row", "col"])
    .dropna(subset="north")
)
df_cc_1["card_count"] = (
    df_cc_1[["north", "east", "west", "south"]].astype(int).sum(axis=1)
)
df_cc_1 = df_cc_1[df_cc_1.card_count == 1]
df_cc_1["file_ok"] = df_cc_1["file_name"].apply(
    lambda x: PATH_TO_IMAGES.joinpath(x).is_file()
)
df_cc_1 = df_cc_1[df_cc_1.file_ok == True]
df_cc_1["leaf_id"] = (
    df_cc_1.plate.astype(str) + df_cc_1.row.astype(str) + df_cc_1.col.astype(str)
)
df_cc_1["leaf_pos"] = df_cc_1.row.astype(str) + df_cc_1.col.astype(str)
df_cc_1 = df_cc_1[~df_cc_1.leaf_pos.isin(["1A", "1B", "1C"])]
df_cc_1

In [ ]:
sel_merge_plate = pn.widgets.Select(
    name="Plate",
    options=list(df.plate.unique()),
    sizing_mode="scale_width",
    value="P04",
)
sel_merge_row = pn.widgets.Select(
    name="Row", options=list(df.row.unique()), sizing_mode="scale_width", value=7
)
sel_merge_col = pn.widgets.Select(
    name="Col", options=list(df.col.unique()), sizing_mode="scale_width", value="G"
)

bt_random = pn.widgets.Button(name="Random Disc")

img_merge_out = pn.pane.Image(sizing_mode="scale_width")
img_single_merge = pn.pane.Image(sizing_mode="scale_width")
img_channel_merge = pn.pane.Image(sizing_mode="scale_width")
img_diff_merge = pn.pane.Image(sizing_mode="scale_width")


updating = False


def filter_df() -> pd.DataFrame:
    return df_cc_1[
        (df_cc_1.plate == sel_merge_plate.value)
        & (df_cc_1.col == sel_merge_col.value)
        & (df_cc_1.row == sel_merge_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]

        sel_merge_plate.value = row.plate
        sel_merge_row.value = row.row
    finally:
        updating = False
    sel_merge_col.value = row.col


bt_random.on_click(on_random)

merge_method = ImageMergeMethod.RGB.value


@pn.depends(
    *[
        w.param.value
        for w in [sel_merge_plate, sel_merge_row, sel_merge_col]
    ],
    watch=True,
)
def on_ld_changed(plate, row, col):
    if updating is True:
        return
    df_ld = filter_df()
    first_image = load(df_ld.iloc[0])
    height, width, _ = first_image.shape
    circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    image_list = [
        crop_image(load(row[1]), crop_data)
        for row in df_ld[df_ld.light_cycle == "ONE_FOURTH"].iterrows()
    ]
    img_merge_out.object = to_pil(concat_tile_resize([image_list]))
    img_cm = merge_images_channels(
        image_list=image_list, color_space=merge_method[0], merge_modes=merge_method[1]
    )
    img_sm = merge_images(image_list=image_list, merge_mode=ImageMergeMode.MIN)
    img_channel_merge.object = to_pil(img_cm)
    img_single_merge.object = to_pil(img_sm)
    img_diff_merge.object = to_pil(np.abs(img_sm - img_cm))


on_ld_changed(
    sel_merge_plate.value, sel_merge_row.value, sel_merge_col.value
)

pn.Column(
    pn.Row(sel_merge_plate, sel_merge_row, sel_merge_col),
    pn.Row(img_merge_out),
    pn.Row(img_channel_merge, img_single_merge, img_diff_merge),
)

In [ ]:
path_to_csv = PATH_TO_PPIMAGES.parent.joinpath(f"{EXP}_{INOC}").with_suffix(".csv")

if path_to_csv.is_file() is False:
    merge_method = ImageMergeMethod.RGB.value
    data = []
    ensure_folder(PATH_TO_PPIMAGES)

    data = {"cycle_id": [], "cx": [], "cy": [], "radius": []}
    errors = []

    for cycle_id in tqdm(df_cc_1.cycle_id.unique()):
        df_ = df_cc_1[df_cc_1.cycle_id == cycle_id]
        first_image = load(df_.iloc[0])
        height, width, _ = first_image.shape
        circles = get_circles(first_image, color_space="hsv", channel="s", resize_factor=8)
        if len(circles["accepted"]) == 1:
            _, cx, cy, r = circles["accepted"][0]
        else:
            errors.append(df_)
            continue
        data["cycle_id"].append(cycle_id)
        data["cx"].append(cx)
        data["cy"].append(cy)
        data["radius"].append(r)
    df_circles = pd.DataFrame(data=data)

    df_pre_process = df_circles.merge(
        df_cc_1.assign(cycle_id=df_cc_1.cycle_id.astype(np.int64)),
        how="left",
        on="cycle_id",
    ).assign(radius=lambda x: x.radius.max() + 16)
    df_pre_process

    for cycle_id in tqdm(df_pre_process.cycle_id.unique()):
        df_ = df_pre_process[df_pre_process.cycle_id == cycle_id]
        row = df_.iloc[0]
        crop_data = Rectangle.from_circle((row.cx, row.cy, row.radius))
        image_list = [crop_image(load(row[1]), crop_data) for row in df_.iterrows()]
        df_leaf_merged = (
            df_.drop(
                [
                    "cx",
                    "cy",
                    "radius",
                    "date_time",
                    "file_name",
                    "date",
                    "year",
                    "month",
                    "day",
                    "time",
                    "hour",
                    "minute",
                    "second",
                    "north",
                    "east",
                    "west",
                    "south",
                    "light_cycle",
                    "center_on_leaf",
                    "crop_top",
                    "crop_bottom",
                    "crop_left",
                    "crop_right",
                    "card_count",
                    "file_ok",
                    "leaf_id",
                    "leaf_pos",
                ],
                axis=1,
            )
            .groupby(["exp", "inoc", "plate", "row", "col", "cycle_id"])
            .mean()
            .reset_index()
            .assign(
                plate=lambda x: x.plate.astype(str)
                .str.replace("P0", "")
                .str.replace("P", "")
                .astype(int)
            )
            .assign(inoc=lambda x: x.inoc.astype(str).str.replace("I", "").astype(int))
            .assign(
                file_name=lambda x: x.exp
                + "#I"
                + x.inoc.astype(str)
                + "#P"
                + x.plate.astype(str)
                + "#R"
                + x.row.astype(str)
                + "#C"
                + x.col.astype(str)
                + "#"
                + x.cycle_id.astype(str)
            )
        )
        data.append(df_leaf_merged)
        # pprint(df_leaf_merged)
        # pprint(df_leaf_merged.iloc[0].file_name)
        cv2.imwrite(
            str(
                PATH_TO_PPIMAGES.joinpath(df_leaf_merged.iloc[0].file_name).with_suffix(
                    ".png"
                )
            ),
            cv2.cvtColor(
                merge_images_channels(
                    image_list=image_list,
                    color_space=merge_method[0],
                    merge_modes=merge_method[1],
                ),
                cv2.COLOR_RGB2BGR,
            ),
        )
    df_final = pd.concat(data)
    date_time = pd.to_datetime(df_final.cycle_id, format=TIME_FORMAT)
    df_final["date_time"] = date_time
    df_final["date"] = date_time.dt.date
    df_final["year"] = date_time.dt.year
    df_final["month"] = date_time.dt.month
    df_final["day"] = date_time.dt.day
    df_final["time"] = date_time.dt.time
    df_final["hour"] = date_time.dt.hour
    df_final["minute"] = date_time.dt.minute
    df_final["second"] = date_time.dt.second
    write_dataframe(df_final, path_to_csv)
else:
    df_final = read_dataframe(path_to_csv)

In [ ]:
df_sample = (
    df_final.sample(n=4)
    .assign(plate=lambda x: x.plate.astype(int))
    .sort_values(["date","plate"])
)

plot_images_with_histograms(
    images=[
        load_image(PATH_TO_PPIMAGES.joinpath(row.file_name).with_suffix(".png"))
        for row in df_sample.itertuples()
    ],
    titles=[
        f"{row.date} - {row.plate}, {row.row}-{row.col} "
        for row in df_sample.itertuples()
    ],
)

In [ ]:
job_ts = df_final.sample(n=1).iloc[0].job_ts
df_plate = df_final[df_final.job_ts == job_ts]

fig = Figure(figsize=(18, 18))
axii = fig.subplots(nrows=9, ncols=9)

columns = df_final.col.sort_values().unique()
rows = df_final.row.sort_values().unique()

pprint(df_plate[["plate", "date"]].sample(n=1))

for c, col in enumerate(columns):
    for r, row in enumerate(rows):
        df_ = df_plate[(df_plate.col == col) & (df_plate.row == row)]
        if len(df_) > 0:
            row = df_.iloc[0]
            axii[c, r].imshow(
                cv2.rotate(
                    load_image(
                        PATH_TO_PPIMAGES.joinpath(row.file_name).with_suffix(".png")
                    ),
                    cv2.ROTATE_90_CLOCKWISE,
                )
            )
        axii[c, r].set_axis_off()

fig.tight_layout()
fig.subplots_adjust(wspace=0, hspace=0)
fig